# LangChain 核心模块学习：Model I/O

`Model I/O` 是 LangChain 为开发者提供的一套面向 LLM 的标准化模型接口，包括模型输入（Prompts）、模型输出（Output Parsers）和模型本身（Models）。

- Prompts：模板化、动态选择和管理模型输入
- Models：以通用接口调用语言模型
- Output Parser：从模型输出中提取信息，并规范化内容

不要混用新旧版本：LangChain 1.0 是一次不兼容更新，langchain-core 从 0.x 升到 1.x 后，旧的 langchain-community 等包必须同步升级，否则必然冲突。
优先用虚拟环境：LangChain 依赖复杂，不同项目的版本要求差异大，用虚拟环境可以避免互相干扰。

## 模型抽象 Model

- 语言模型(LLMs): LangChain 的核心组件。LangChain并不提供自己的LLMs，而是为与许多不同的LLMs（OpenAI、Cohere、Hugging Face等）进行交互提供了一个标准接口。
    OpenAI Completion API（旧版）
    文本补全模型
- 聊天模型(Chat Models): 语言模型的一种变体。虽然聊天模型在内部使用了语言模型，但它们提供的接口略有不同。与其暴露一个“输入文本，输出文本”的API不同，它们提供了一个以“聊天消息”作为输入和输出的接口。

（注：对比 OpenAI Completion API和 Chat Completion API）
```
BaseLanguageModel  # 顶层祖宗类：所有模型都继承它
├── BaseLLM         # 文本补全模型基类
│     └── LLM
│         └── 各种文本模型（OpenAI/智谱/通义）
└── BaseChatModel   # 聊天模型基类
      └── ChatOpenAI、ChatZhipuAI、ChatTongyi
```

## 语言模型（LLMs)

类继承关系：

```
BaseLanguageModel --> BaseLLM --> LLM --> <name>  # Examples: AI21, HuggingFaceHub, OpenAI
```

主要抽象:

```
LLMResult, PromptValue,
CallbackManagerForLLMRun, AsyncCallbackManagerForLLMRun,
CallbackManager, AsyncCallbackManager,
AIMessage, BaseMessage
```

核心方法:
    所有模型都有这 3 套方法：
    invoke() → 直接调用，返回结果
    stream() → 流式输出（打字机效果）
    batch() → 批量调用
    统一接口，换模型不换代码！


### BaseLanguageModel Class

这个基类为语言模型定义了一个接口，该接口允许用户以不同的方式与模型交互（例如通过提示或消息）。`generate_prompt` 是其中的一个主要方法，它接受一系列提示，并返回模型的生成结果。

```python
# 定义 BaseLanguageModel 抽象基类，它从 Serializable, Runnable 和 ABC 继承
class BaseLanguageModel(
    Serializable, Runnable[LanguageModelInput, LanguageModelOutput], ABC
):
    """
    与语言模型交互的抽象基类。
    所有语言模型的封装器都应从 BaseLanguageModel 继承。
    
    Serializable = 可序列化
        这个类可以被保存、传递、日志记录。
    Runnable = 可运行
        可以被调用：invoke ()、stream ()、batch ()
    ABC = 抽象基类
        所有继承它的模型，必须按规矩实现方法。

    这个类主要提供三种方法：
    - generate_prompt: 为一系列的提示值生成语言模型输出。提示值是可以转换为任何语言模型输入格式的模型输入（如字符串或消息）。
    - predict: 将单个字符串传递给语言模型并返回字符串预测。
    - predict_messages: 将一系列 BaseMessages（对应于单个模型调用）传递给语言模型，并返回 BaseMessage 预测。

    每种方法都有对应的异步方法。
    """

    # 定义一个抽象方法 generate_prompt，必须由子类实现！ 统一入口，屏蔽不同模型差异，实现统一调用。
    @abstractmethod
    def generate_prompt(
        self,
        prompts: List[PromptValue],  # 输入提示的列表
        stop: Optional[List[str]] = None,  # 生成时的停止词列表
        callbacks: Callbacks = None,  # 回调，用于执行例如日志记录或流式处理的额外功能
        **kwargs: Any,  # 任意的额外关键字参数，通常会传递给模型提供者的 API 调用
    ) -> LLMResult:
        """
        将一系列的提示传递给模型并返回模型的生成。
        对于提供批处理 API 的模型，此方法应使用批处理调用。

        使用此方法时：
            1. 希望利用批处理调用，
            2. 需要从模型中获取的输出不仅仅是最顶部生成的值，
            3. 构建与底层语言模型类型无关的链（例如，纯文本完成模型与聊天模型）。

        参数:
            prompts: 提示值的列表。提示值是一个可以转换为与任何语言模型匹配的格式的对象（对于纯文本生成模型为字符串，对于聊天模型为 BaseMessages）。
              PromptValue 的作用：
                自动适配不同模型的输入格式，你不用管模型是文本还是聊天，它自动转。
            stop: 生成时使用的停止词。模型看到这些词，就停止生成。比如 stop=["结束", "停止"]，生成到这两个词就停。
            callbacks: 要传递的回调。用于执行额外功能，例如在生成过程中进行日志记录或流式处理。
                打印日志
                实现流式输出（打字机效果）
                监控、报错、追踪
                LangSmith 调试
            **kwargs:透传参数， 任意的额外关键字参数。通常这些会传递给模型提供者的 API 调用。
            比如：temperature、top_p、max_tokens、这些模型参数直接传给底层 API。

        返回值:
            LLMResult，返回值：标准格式的结果。它包含每个输入提示的候选生成列表以及特定于模型提供者的额外输出。
            里面包含：
                生成的文本 / 消息
                token 使用量
                模型额外信息
            不管什么模型，返回格式都一样！
        """
```


### BaseLLM Class 专门做「文本补全」的基类


这段代码定义了一个名为 BaseLLM 的抽象基类。这个基类的主要目的是提供一个基本的接口来处理大型语言模型 (LLM)。

它的特点：
输入 = 一段字符串（prompt）
输出 = 一段字符串（completion）
场景 = 续写、填空、总结、翻译、文本生成
不支持聊天消息，只支持纯文本

```python
# 定义 BaseLLM 抽象基类，它从 BaseLanguageModel[str] 和 ABC（Abstract Base Class）继承
class BaseLLM(BaseLanguageModel[str], ABC):
    """
     [str] 表示：输入输出都是字符串（string）
    
    """

    # 是否开启缓存，其初始值为 None
    # 开启后，相同 prompt 不会重复调用模型，直接返回结果→ 省钱、加速
    cache: Optional[bool] = None

    # 定义 verbose 属性，该属性决定是否打印响应文本，是否打印模型调用过程，开发调试用，True 会输出详细过程
    # 默认值使用 _get_verbosity 函数的结果
    verbose: bool = Field(default_factory=_get_verbosity)
   

    # 定义 callbacks 属性，其初始值为 None，并从序列化中排除
    # 做流式输出、日志、监控、报错处理
    callbacks: Callbacks = Field(default=None, exclude=True)

    # 定义 callback_manager 属性，其初始值为 None，并从序列化中排除
    callback_manager: Optional[BaseCallbackManager] = Field(default=None, exclude=True)

    # 定义 tags 属性，这些标签会被添加到运行追踪中，其初始值为 None，并从序列化中排除
    # 给模型调用打标签、加信息，给 LangSmith 调试平台看的
    tags: Optional[List[str]] = Field(default=None, exclude=True)
    

    # 定义 metadata 属性，这些元数据会被添加到运行追踪中，其初始值为 None，并从序列化中排除
    # 给模型调用打标签、加信息，给 LangSmith 调试平台看的
    metadata: Optional[Dict[str, Any]] = Field(default=None, exclude=True)
    """Metadata to add to the run trace."""

    # 内部类定义了这个 pydantic 对象的配置
    class Config:
        """Configuration for this pydantic object."""

        # 允许使用任意类型
        arbitrary_types_allowed = True

```
这个基类使用了 Pydantic 的功能，特别是 Field 方法，用于定义默认值和序列化行为。BaseLLM 的子类需要提供实现具体功能的方法。

### LLM Class


BaseLLM：只定规矩，很抽象
LLM：帮你实现了 _generate 复杂逻辑

这段代码定义了一个名为 LLM 的类，该类继承自 BaseLLM。这个类的目的是为了为用户提供一个简化的接口来处理LLM（大型语言模型），而不期望用户实现完整的 _generate 方法。


```
BaseLanguageModel
   ↓
BaseLLM
   ↓
LLM （帮你实现了 _generate 批量处理）
   ↓
你写的模型（只需要实现 _call）
_generate 自己没有任何模型逻辑
它循环调用你写的 _call 方法

```


```python

# 继承自 BaseLLM 的 LLM 类
class LLM(BaseLLM):
    """Base LLM abstract class.

    The purpose of this class is to expose a simpler interface for working
    with LLMs, rather than expect the user to implement the full _generate method.
    """

    # 使用 @abstractmethod 装饰器定义一个抽象方法，子类需要实现这个方法
    @abstractmethod
    def _call(
        self,
        prompt: str,  # 输入提示
        stop: Optional[List[str]] = None,  # 停止词列表
        run_manager: Optional[CallbackManagerForLLMRun] = None,  # 运行管理器
        **kwargs: Any,  # 其他关键字参数
    ) -> str:
        """Run the LLM on the given prompt and input."""
        # 此方法的实现应在子类中提供

    
    # _generate 方法使用了 _call 方法，用于处理多个提示。LangChain 最核心的设计之一
    # 处理一批（多个）提示词，批量调用模型
    def _generate(
        self,
        prompts: List[str],  # 多个输入提示的列表，一堆提示词 [prompt1, prompt2, prompt3]
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> LLMResult:
        """Run the LLM on the given prompt and input."""
        generations = []  # 用于存储生成的文本
        # 检查 _call 方法的签名是否支持 run_manager 参数
        new_arg_supported = inspect.signature(self._call).parameters.get("run_manager")
        ## 循环每一个 prompt
        for prompt in prompts:  # 遍历每个提示
            # 根据是否支持 run_manager 参数来选择调用方法
            text = (
                self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
                if new_arg_supported
                else self._call(prompt, stop=stop, **kwargs)
            )
            # 将生成的文本添加到 generations 列表中
            generations.append([Generation(text=text)])
        # 返回 LLMResult 对象，其中包含 generations 列表
        return LLMResult(generations=generations)
```

### LLMs 已支持模型清单

**已支持模型清单地址：https://docs.langchain.com/oss/python/integrations/chat


```
BaseLanguageModel
   ↓
BaseLLM
   ↓
LLM
   ↓
BaseOpenAI  （你现在看的这个）
```
### 使用 LangChain 调用 OpenAI GPT Completion API

#### BaseOpenAI Class

```python
class BaseOpenAI(BaseLLM):
    """
    OpenAI 大语言模型的基类。
    继承 BaseLLM
    定义模型参数（model、temperature、max_tokens...）
    实现 _call 方法：输入字符串 → 输出字符串
    自动拥有 _generate 批量能力
    """

    @property
    def lc_secrets(self) -> Dict[str, str]:
        return {"openai_api_key": "OPENAI_API_KEY"}

    @property
    def lc_serializable(self) -> bool:
        return True

    client: Any  #: :meta private:
    model_name: str = Field("text-davinci-003", alias="model")
    """使用的模型名。"""
    temperature: float = 0.7
    """要使用的采样温度。"""
    max_tokens: int = 256
    """完成中生成的最大令牌数。 
    -1表示根据提示和模型的最大上下文大小返回尽可能多的令牌。"""
    top_p: float = 1
    """在每一步考虑的令牌的总概率质量。"""
    frequency_penalty: float = 0
    """根据频率惩罚重复的令牌。"""
    presence_penalty: float = 0
    """惩罚重复的令牌。"""
    n: int = 1
    """为每个提示生成多少完成。"""
    best_of: int = 1
    """在服务器端生成best_of完成并返回“最佳”。"""
    model_kwargs: Dict[str, Any] = Field(default_factory=dict)
    """保存任何未明确指定的`create`调用的有效模型参数。"""
    openai_api_key: Optional[str] = None
    openai_api_base: Optional[str] = None
    openai_organization: Optional[str] = None
    # 支持OpenAI的显式代理
    openai_proxy: Optional[str] = None
    # → 批量大小、请求超时、最大重试次数
    batch_size: int = 20
    """传递多个文档以生成时使用的批处理大小。"""
    request_timeout: Optional[Union[float, Tuple[float, float]]] = None
    """向OpenAI完成API的请求超时。 默认为600秒。"""
    logit_bias: Optional[Dict[str, float]] = Field(default_factory=dict)
    """调整生成特定令牌的概率。"""
    max_retries: int = 6
    """生成时尝试的最大次数。"""
    streaming: bool = False
    """是否流式传输结果。"""
    allowed_special: Union[Literal["all"], AbstractSet[str]] = set()
    """允许的特殊令牌集。"""
    disallowed_special: Union[Literal["all"], Collection[str]] = "all"
    """不允许的特殊令牌集。"""
    tiktoken_model_name: Optional[str] = None
    """使用此类时传递给tiktoken的模型名。
    Tiktoken用于计算文档中的令牌数量以限制它们在某个限制以下。
    默认情况下，设置为None时，这将与嵌入模型名称相同。
    但是，在某些情况下，您可能希望使用此嵌入类与tiktoken不支持的模型名称。
    这可以包括使用Azure嵌入或使用多个模型提供商的情况，这些提供商公开了类似OpenAI的API但模型不同。
    在这些情况下，为了避免在调用tiktoken时出错，您可以在此处指定要使用的模型名称。"""
```

```
# OpenAI _call 核心逻辑
def _call(self, prompt: str, stop: List[str] = None) -> str:
    response = openai.Completion.create(
        model=self.model_name,
        prompt=prompt,
        temperature=self.temperature,
        max_tokens=self.max_tokens
    )
    return response.choices[0].text
```